# RAG Evaluation Arena

Metrics:
1. **ROUGE-L** - Textual overlap
2. **LLM-as-Judge** - Faithfulness & Accuracy (1-5)
3. **Latency** - Response time (ms)

## 1. Setup

In [1]:
import sys, os, json, time, re
from pathlib import Path
from typing import List, Dict
from dotenv import load_dotenv

notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == "notebooks" else notebook_dir
os.chdir(project_root)
sys.path.insert(0, str(project_root))
load_dotenv()

from src.services.llm_services import load_config, get_llm, get_text_embeddings
config = load_config("src/config/config.yaml")
print("Config loaded!")

Config loaded!


In [1]:
%pip install rouge-score

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Load Golden Test Set

In [2]:
test_set = []
with open("data/qa_pairs/golden_test_set.jsonl", "r") as f:
    for line in f:
        test_set.append(json.loads(line.strip()))

print(f"Loaded {len(test_set)} test questions")
print(f"Sample Q: {test_set[0]['question']}")
print(f"Sample A: {test_set[0]['answer']}")

Loaded 636 test questions
Sample Q: How did Mobility Gross Bookings change year-over-year in 2024, on a constant currency basis?
Sample A: Mobility Gross Bookings grew 25% year-over-year, on a constant currency basis.


## 3. Setup RAG Pipeline

In [3]:
import gc
import torch
import os

# Clear memory before loading new models
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU Memory: {torch.cuda.memory_allocated()/1024**3:.2f}GB allocated")

import weaviate

# Connect to Weaviate - use API key from .env
cloud_config = config["weaviate"]["cloud"]
weaviate_api_key = os.getenv("WEAVIATE_API_KEY")

if not weaviate_api_key:
    raise ValueError("WEAVIATE_API_KEY not found in .env file!")

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=cloud_config["cluster_url"],
    auth_credentials=weaviate.auth.AuthApiKey(weaviate_api_key),
)
collection = client.collections.get(config["weaviate"]["schema"]["class_name"])

# Load embeddings and LLM
embeddings = get_text_embeddings(config)
llm = get_llm(config)

# Lazy load reranker to save memory
reranker = None

def get_reranker():
    global reranker
    if reranker is None:
        from sentence_transformers import CrossEncoder
        reranker = CrossEncoder(config["retrieval"]["reranker_model"], max_length=512)
    return reranker

print("RAG pipeline ready!")

c:\Users\viraj\Zuu\Ledger_mind\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\viraj\Zuu\Ledger_mind\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\viraj\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In ord

RAG pipeline ready!


In [4]:
def hybrid_search(query: str, top_k: int = 20, alpha: float = 0.6) -> List[Dict]:
    query_vector = embeddings.embed_query(query)
    response = collection.query.hybrid(
        query=query, vector=query_vector, alpha=alpha, limit=top_k,
        return_metadata=["score"], return_properties=["content", "chunk_index"]
    )
    return [{"content": o.properties["content"], "chunk_index": o.properties["chunk_index"],
             "score": o.metadata.score or 0} for o in response.objects]

def rerank(query: str, docs: List[Dict], top_n: int = 3) -> List[Dict]:
    if not docs: return []
    rr = get_reranker()  # Lazy load
    pairs = [(query, d["content"]) for d in docs]
    scores = rr.predict(pairs, show_progress_bar=False)
    for d, s in zip(docs, scores): d["rerank_score"] = float(s)
    return sorted(docs, key=lambda x: x["rerank_score"], reverse=True)[:top_n]

def rag_query(question: str, top_k: int = 20, top_n: int = 3) -> Dict:
    start = time.time()
    docs = hybrid_search(question, top_k=top_k)
    if not docs:
        return {"answer": "No info found.", "latency_ms": (time.time()-start)*1000, "sources": []}
    top_docs = rerank(question, docs, top_n=top_n)
    context = "\n\n".join([f"[Doc {i+1}]\n{d['content']}" for i, d in enumerate(top_docs)])
    prompt = f"Answer based on context. Be concise.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    resp = llm.invoke(prompt)
    answer = resp.content if hasattr(resp, "content") else str(resp)
    return {"answer": answer, "latency_ms": (time.time()-start)*1000, "sources": [d["chunk_index"] for d in top_docs]}

print("RAG functions ready!")

RAG functions ready!


## 4. ROUGE-L Metric

In [5]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def compute_rouge_l(prediction: str, reference: str) -> float:
    scores = scorer.score(reference, prediction)
    return scores["rougeL"].fmeasure

print(f"ROUGE-L test: {compute_rouge_l('Uber moves people', 'Uber helps people move'):.3f}")

ROUGE-L test: 0.571


## 5. LLM-as-Judge

In [6]:
def llm_judge(question: str, ground_truth: str, generated: str) -> Dict:
    prompt = f"""Score the generated answer (1-5).

Question: {question}
Ground Truth: {ground_truth}
Generated: {generated}

Reply ONLY as:
Faithfulness: [score]
Accuracy: [score]"""
    
    resp = llm.invoke(prompt)
    text = resp.content if hasattr(resp, "content") else str(resp)
    
    faith = re.search(r"Faithfulness:\s*(\d)", text)
    acc = re.search(r"Accuracy:\s*(\d)", text)
    
    return {
        "faithfulness": int(faith.group(1)) if faith else 3,
        "accuracy": int(acc.group(1)) if acc else 3
    }

print("LLM Judge ready!")

LLM Judge ready!


## 6. Run Evaluation

In [7]:
from tqdm import tqdm

def evaluate_rag(samples: List[Dict], n: int = None) -> List[Dict]:
    if n: samples = samples[:n]
    results = []
    for s in tqdm(samples, desc="Evaluating"):
        rag_out = rag_query(s["question"])
        judge = llm_judge(s["question"], s["answer"], rag_out["answer"])
        results.append({
            "question": s["question"],
            "ground_truth": s["answer"],
            "generated": rag_out["answer"],
            "rouge_l": compute_rouge_l(rag_out["answer"], s["answer"]),
            "faithfulness": judge["faithfulness"],
            "accuracy": judge["accuracy"],
            "latency_ms": rag_out["latency_ms"],
            "chunk_hit": s["chunk_index"] in rag_out["sources"],
            "expected_chunk": s["chunk_index"],
            "retrieved": rag_out["sources"]
        })
    return results

print("Ready!")

Ready!


In [8]:
NUM_SAMPLES = 10
results = evaluate_rag(test_set, n=NUM_SAMPLES)

Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]INFO:weaviate-client:Searching in collection UberReportChunk received exception: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "IOCP/Socket: Connection timed out (A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond.
 -- 10060)"
	debug_error_string = "UNKNOWN:Error received from peer  {grpc_message:"IOCP/Socket: Connection timed out (A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond.\r\n -- 10060)", grpc_status:14}"
>. Retrying with exponential backoff in 1 seconds
c:\Users\viraj\Zuu\Ledger_mind\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to effici

## 7. Results Summary

In [9]:
import numpy as np

def summary(results):
    n = len(results)
    print("=" * 50)
    print(f"RESULTS ({n} samples)")
    print("=" * 50)
    print(f"ROUGE-L:      {np.mean([r['rouge_l'] for r in results]):.3f}")
    print(f"Faithfulness: {np.mean([r['faithfulness'] for r in results]):.2f}/5")
    print(f"Accuracy:     {np.mean([r['accuracy'] for r in results]):.2f}/5")
    print(f"Latency:      {np.mean([r['latency_ms'] for r in results]):.0f}ms")
    print(f"Chunk Hit:    {np.mean([r['chunk_hit'] for r in results])*100:.1f}%")
    print("=" * 50)

summary(results)

RESULTS (10 samples)
ROUGE-L:      0.366
Faithfulness: 5.00/5
Accuracy:     3.80/5
Latency:      14995ms
Chunk Hit:    100.0%


In [10]:
import pandas as pd
df = pd.DataFrame(results)[['question', 'rouge_l', 'faithfulness', 'accuracy', 'latency_ms', 'chunk_hit']]
df['question'] = df['question'].str[:40] + '...'
df

,question,rouge_l,faithfulness,accuracy,latency_ms,chunk_hit
0,How did Mobility Gross Bookings change y...,0.742857,5,5,67866.816282,True
1,"According to the document, what are the ...",0.185185,5,2,6001.174450,True
2,In what ways have the company's operatio...,0.155689,5,4,18141.349077,True
3,In what month and year did the European ...,0.307692,5,5,24002.531290,True
4,What types of security breaches may our ...,0.482759,5,5,4810.350180,True
5,What are the potential consequences of f...,0.750000,5,5,6696.961641,True
6,What is the name of the website that con...,0.222222,5,5,7840.976000,True
7,"According to the document, what are some...",0.230769,5,1,3529.186010,True
8,What are some examples of market shifts ...,0.400000,5,4,7712.512493,True
9,"What information, in addition to drivers...",0.181818,5,2,3349.849224,True


## 8. Save Results

In [11]:
from datetime import datetime

output = {
    "timestamp": datetime.now().isoformat(),
    "num_samples": len(results),
    "summary": {
        "rouge_l": float(np.mean([r['rouge_l'] for r in results])),
        "faithfulness": float(np.mean([r['faithfulness'] for r in results])),
        "accuracy": float(np.mean([r['accuracy'] for r in results])),
        "latency_ms": float(np.mean([r['latency_ms'] for r in results])),
        "chunk_hit_rate": float(np.mean([r['chunk_hit'] for r in results]))
    },
    "results": results
}

Path("artifacts").mkdir(exist_ok=True)
with open("artifacts/evaluation_results.json", "w") as f:
    json.dump(output, f, indent=2)
print("Saved to artifacts/evaluation_results.json")

Saved to artifacts/evaluation_results.json


## 9. Cleanup

In [12]:
client.close()
print("Done!")

Done!
